# QK Circuit Analysis

Deep analysis of query-key (QK) circuits: eigenvalue structure,
positional vs content contributions, pattern prediction from weights,
inter-layer QK composition, and effective receptive field.

The QK circuit (W_Q @ W_K^T) determines what each head attends to.

## Why This Matters

QK circuit circuit analyzes the query-key interaction that determines attention patterns. Understanding what features drive attention — positional vs. content-based, local vs. global — reveals how the model decides which information to route where.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.qk_circuit_analysis import (
    qk_eigenvalue_structure,
    positional_vs_content_qk,
    qk_pattern_prediction,
    qk_composition_analysis,
    effective_receptive_field,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## QK Eigenvalue Structure

In [ ]:
for l in range(min(2, cfg.n_layers)):
    for h in range(min(2, cfg.n_heads)):
        result = qk_eigenvalue_structure(model, layer=l, head=h)
        print(f"L{l}H{h}: trace={result['qk_trace']:.4f}, rank={result['qk_rank']:.1f}, "
              f"+/- ratio={result['positive_negative_ratio']:.2f}")
        print(f"  Top eigenvalues: {[f'{e:.3f}' for e in result['eigenvalues'][:3]]}")

## Positional vs Content QK

In [ ]:
for l in range(min(2, cfg.n_layers)):
    for h in range(min(2, cfg.n_heads)):
        result = positional_vs_content_qk(model, tokens, layer=l, head=h)
        print(f"L{l}H{h}: content={result['content_fraction']:.1%}, "
              f"positional={result['positional_fraction']:.1%}")

## QK Pattern Prediction

How well can we predict attention patterns from weights alone (layer 0)?

In [ ]:
for h in range(cfg.n_heads):
    result = qk_pattern_prediction(model, tokens, layer=0, head=h)
    print(f"L0H{h}: correlation={result['correlation']:.4f}, MSE={result['mse']:.6f}, "
          f"max_error at {result['max_error_position']}")

## QK Composition

In [ ]:
for sh in range(min(2, cfg.n_heads)):
    for dh in range(min(2, cfg.n_heads)):
        result = qk_composition_analysis(model, 0, sh, 1, dh)
        print(f"L0H{sh}->L1H{dh}: Q={result['q_composition_score']:.3f}, "
              f"K={result['k_composition_score']:.3f}, strength={result['composition_strength']:.3f}")

## Effective Receptive Field

In [ ]:
for l in range(min(2, cfg.n_layers)):
    for h in range(min(2, cfg.n_heads)):
        result = effective_receptive_field(model, tokens, layer=l, head=h)
        print(f"L{l}H{h}: mean_dist={result['mean_attention_distance']:.2f}, "
              f"width={result['receptive_field_width']:.2f}, peaks={result['peak_positions'].tolist()}")